# 14 — Abstractive Cluster Descriptions (FLAN-T5-large)

Replaces LexRank extractive summaries with abstractive descriptions generated by FLAN-T5-large.

**Flow per cluster:**
1. Random-sample up to 100 summaries (seed=42)
2. Batch into groups of ~15 summaries (≤512 tokens each)
3. FLAN-T5 summarizes each batch → mini-summaries
4. FLAN-T5 final pass over mini-summaries → 2-3 general sentences
5. Save per-method JSON + full human-readable markdown report

**Section 2:** Re-run semantic category mapping with new descriptions  
**Section 3:** Compare semantic scores — LexRank vs Abstractive

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json
import random
import textwrap
import numpy as np
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer

from utils import config

# ── Data ──────────────────────────────────────────────────────────
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
summaries   = train_clean["summary"].tolist()

label_files = sorted(config.RESULTS_DIR.glob("cluster_labels_*.npy"))
ALL_METHODS = [f.stem.replace("cluster_labels_", "") for f in label_files]
print(f"Found {len(ALL_METHODS)} clustering methods")

SAMPLE_SIZE  = 100
BATCH_SIZE   = 15   # summaries per first-pass batch
RANDOM_SEED  = 42

In [ ]:
# ── Load FLAN-T5-large ────────────────────────────────────────────
print("Loading google/flan-t5-large ...")
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model     = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")
model.eval()
print("Model loaded.")

In [ ]:
import torch

def t5_summarize(text: str, max_new_tokens: int = 128) -> str:
    """Run one T5 inference call. Returns decoded output string."""
    prompt = (
        "Summarize the main topic of the following news articles "
        "in 1-2 general sentences, without mentioning specific names or events:\n\n"
        + text
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    )
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def abstractive_description(cluster_summaries: list[str]) -> str:
    """
    Given a list of article summaries for one cluster:
    1. Random-sample up to SAMPLE_SIZE
    2. Batch into BATCH_SIZE groups → first-pass mini-summaries
    3. Final pass over mini-summaries → 2-3 general sentences
    """
    rng = random.Random(RANDOM_SEED)
    sample = rng.sample(cluster_summaries, min(SAMPLE_SIZE, len(cluster_summaries)))

    # First pass
    mini_summaries = []
    for i in range(0, len(sample), BATCH_SIZE):
        batch = sample[i : i + BATCH_SIZE]
        batch_text = "\n- ".join(batch)
        mini_summaries.append(t5_summarize(batch_text, max_new_tokens=80))

    # Final pass
    if len(mini_summaries) == 1:
        return mini_summaries[0]

    final_prompt = (
        "Summarize the main topic of the following news articles "
        "in 2-3 general sentences, without mentioning specific names or events:\n\n"
        + "\n".join(mini_summaries)
    )
    inputs = tokenizer(
        final_prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    )
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print("Helper functions defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# SECTION 1 — Generate abstractive descriptions for all methods
# ════════════════════════════════════════════════════════════════════
from tqdm.auto import tqdm

all_descriptions = {}   # method -> {cluster_id(int): description(str)}

for method in ALL_METHODS:
    out_path = config.RESULTS_DIR / f"cluster_descriptions_abstractive_{method}.json"

    if out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            raw = json.load(f)
        all_descriptions[method] = {int(k): v for k, v in raw.items()}
        print(f"  {method}: loaded from cache ({len(all_descriptions[method])} clusters)")
        continue

    cluster_labels = np.load(config.RESULTS_DIR / f"cluster_labels_{method}.npy")
    unique_clusters = sorted(set(cluster_labels))
    descs = {}

    for cid in tqdm(unique_clusters, desc=method, leave=False):
        idx = np.where(cluster_labels == cid)[0]
        cluster_sums = [summaries[i] for i in idx]
        descs[cid] = abstractive_description(cluster_sums)

    all_descriptions[method] = descs
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in descs.items()}, f, indent=2, ensure_ascii=False)
    print(f"  {method}: {len(descs)} descriptions saved → {out_path.name}")

print(f"\nSection 1 complete. {len(all_descriptions)} methods processed.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Save human-readable markdown report (full version)
# ════════════════════════════════════════════════════════════════════
report_path = config.RESULTS_DIR / "abstractive_descriptions_report.md"

lines = ["# Abstractive Cluster Descriptions Report\n",
         "> Generated by FLAN-T5-large | Top-100 random-sampled summaries per cluster\n"]

for method in ALL_METHODS:
    descs = all_descriptions.get(method, {})
    cluster_labels = np.load(config.RESULTS_DIR / f"cluster_labels_{method}.npy")
    clusters_df    = pd.read_csv(config.RESULTS_DIR / f"clusters_{method}.csv")
    majority_map   = dict(zip(clusters_df["cluster"], clusters_df["majority_true_label"]))

    lines.append(f"\n---\n## Method: `{method}`\n")
    for cid in sorted(descs.keys()):
        true_label = majority_map.get(cid, "?")
        n_articles = int(np.sum(cluster_labels == cid))
        lines.append(f"### Cluster {cid} | True label: **{true_label}** | {n_articles} articles\n")
        lines.append(descs[cid] + "\n")

with open(report_path, "w", encoding="utf-8") as f:
    f.writelines(lines)

print(f"Report saved → {report_path}")
print(f"Total size: {report_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Quick preview — first 3 clusters of best method
preview_method = "mpnet_kmeans"
clusters_df  = pd.read_csv(config.RESULTS_DIR / f"clusters_{preview_method}.csv")
majority_map = dict(zip(clusters_df["cluster"], clusters_df["majority_true_label"]))

print(f"=== Preview: {preview_method} ===\n")
for cid in sorted(all_descriptions[preview_method].keys())[:5]:
    true = majority_map.get(cid, "?")
    desc = all_descriptions[preview_method][cid]
    print(f"[Cluster {cid} | {true}]")
    print(textwrap.fill(desc, width=90))
    print()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# SECTION 2 — Semantic category mapping (same as nb13, new descriptions)
# ════════════════════════════════════════════════════════════════════
from sentence_transformers import SentenceTransformer

CANONICAL_DESCRIPTIONS = {
    "ARTS & CULTURE":  "Articles about visual arts, performing arts, music, literature, cultural events, and creative expression.",
    "BUSINESS":        "Articles about corporate finance, markets, economics, trade, and business strategy.",
    "COMEDY":          "Articles about stand-up comedy, satirical shows, late-night television hosts, and humorous commentary on current events.",
    "CRIME":           "Articles about criminal investigations, arrests, court cases, and law enforcement activities.",
    "EDUCATION":       "Articles about schools, universities, student loans, teachers, and educational policy.",
    "ENTERTAINMENT":   "Articles about movies, television shows, celebrity news, awards ceremonies, and pop culture.",
    "ENVIRONMENT":     "Articles about climate change, environmental policy, wildlife, natural disasters, and conservation.",
    "HEALTH":          "Articles about medical research, public health, disease outbreaks, vaccines, and healthcare policy.",
    "MEDIA":           "Articles about journalism, news organisations, television networks, media industry controversies, and press freedom.",
    "NEWS":            "Breaking news and general current events not classified under a more specific category.",
    "POLITICS":        "Articles about elections, government policy, politicians, legislation, and political parties.",
    "RELIGION":        "Articles about religious institutions, faith, spirituality, religious leaders, and interfaith issues.",
    "SCIENCE":         "Articles about scientific discoveries, space exploration, NASA, astronomy, physics, and technology research.",
    "SPORTS":          "Articles about professional and amateur sports competitions, athletes, teams, tournaments, and sporting events.",
    "TECH":            "Articles about consumer technology, gadgets, software, social media platforms, and the tech industry.",
    "WOMEN":           "Articles about women's rights, gender equality, sexual harassment, feminism, and women in public life.",
}

assert list(CANONICAL_DESCRIPTIONS.keys()) == config.CLASS_NAMES

print("Loading all-mpnet-base-v2 for semantic embeddings...")
embed_model    = SentenceTransformer("all-mpnet-base-v2")
category_texts = list(CANONICAL_DESCRIPTIONS.values())
category_names = list(CANONICAL_DESCRIPTIONS.keys())
category_embs  = embed_model.encode(
    category_texts, batch_size=16, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=True,
).astype(np.float32)
print(f"Category embeddings: {category_embs.shape}")

In [ ]:
all_mappings = {}

for method in ALL_METHODS:
    descs       = all_descriptions[method]
    cluster_ids = sorted(descs.keys())
    desc_texts  = [descs[c] for c in cluster_ids]

    cluster_embs = embed_model.encode(
        desc_texts, batch_size=16, show_progress_bar=False,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32)

    sim_matrix = cluster_embs @ category_embs.T

    mapping = {}
    for i, c in enumerate(cluster_ids):
        best_idx = int(np.argmax(sim_matrix[i]))
        mapping[c] = {
            "mapped_category": category_names[best_idx],
            "similarity":      float(sim_matrix[i, best_idx]),
        }
    all_mappings[method] = mapping

    out_path = config.RESULTS_DIR / f"cluster_category_mapping_abstractive_{method}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in mapping.items()}, f, indent=2, ensure_ascii=False)

    print(f"  {method}: mappings saved → {out_path.name}")

print(f"\nSection 2 complete.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# SECTION 3 — Compare LexRank vs Abstractive semantic scores
# ════════════════════════════════════════════════════════════════════
rows = []
for method in ALL_METHODS:
    metrics_path = config.RESULTS_DIR / f"metrics_{method}.json"
    if not metrics_path.exists():
        continue
    metrics = json.load(open(metrics_path))

    clusters_df  = pd.read_csv(config.RESULTS_DIR / f"clusters_{method}.csv")
    majority_map = dict(zip(clusters_df["cluster"], clusters_df["majority_true_label"]))

    # Abstractive scores
    mapping  = all_mappings.get(method, {})
    abs_sims = [v["similarity"] for v in mapping.values()]
    abs_mean = float(np.mean(abs_sims)) if abs_sims else 0.0
    abs_acc  = sum(
        1 for c, v in mapping.items()
        if v["mapped_category"] == majority_map.get(c, "")
    ) / len(mapping) if mapping else 0.0

    # LexRank scores (from existing semantic_evaluation.csv)
    lex_csv  = pd.read_csv(config.RESULTS_DIR / "semantic_evaluation.csv")
    lex_row  = lex_csv[lex_csv["method"] == method]
    lex_mean = float(lex_row["mean_semantic_score"].values[0]) if len(lex_row) else 0.0
    lex_macc = float(lex_row["mapping_accuracy"].values[0])    if len(lex_row) else 0.0

    rows.append({
        "method":           method,
        "ACC_hungarian":    round(metrics["ACC (Hungarian)"], 4),
        "lexrank_sem_score":round(lex_mean, 4),
        "lexrank_map_acc":  round(lex_macc, 4),
        "abs_sem_score":    round(abs_mean, 4),
        "abs_map_acc":      round(abs_acc,  4),
        "sem_score_delta":  round(abs_mean - lex_mean, 4),
    })

compare_df = pd.DataFrame(rows).sort_values("abs_sem_score", ascending=False)
out_csv = config.RESULTS_DIR / "semantic_evaluation_abstractive.csv"
compare_df.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")

In [ ]:
print("\n=== COMPARISON: LexRank vs Abstractive (FLAN-T5-large) ===\n")
print(f"{'Method':<35} {'ACC':>7} {'Lex Sim':>9} {'Abs Sim':>9} {'Delta':>7} {'Abs MapACC':>11}")
print("-" * 85)
for _, row in compare_df.iterrows():
    delta_str = f"+{row['sem_score_delta']:.4f}" if row['sem_score_delta'] >= 0 else f"{row['sem_score_delta']:.4f}"
    print(
        f"{row['method']:<35} {row['ACC_hungarian']:>7.4f} "
        f"{row['lexrank_sem_score']:>9.4f} {row['abs_sem_score']:>9.4f} "
        f"{delta_str:>7} {row['abs_map_acc']:>11.4f}"
    )

print(f"\nBest abstractive semantic score: {compare_df.iloc[0]['method']} ({compare_df.iloc[0]['abs_sem_score']:.4f})")
improved = (compare_df['sem_score_delta'] > 0).sum()
print(f"Methods improved by abstractive: {improved}/{len(compare_df)}")
print(f"Mean delta: {compare_df['sem_score_delta'].mean():+.4f}")